In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
FILIPINO FAKE NEWS DETECTION — HPO TRAINING PIPELINE
Cavite State University · 2026

Models    : Tagalog-BERT (jcblaise/bert-tagalog-base-cased)
            Tagalog-DistilBERT (jcblaise/distilbert-tagalog-base-cased)

Experiment: 2 architectures × N sheets from Excel dataset file
            Sheet naming convention: HR33-AIR67-HF50-AIF50 (encodes ratios)
            Each sheet contains pre-split rows (split ∈ {train, val, test})
            Val / Test sets are shared across all sheets (from first sheet)

HPO       : Optuna TPE · 20 trials · seeds {123, 456, 789} per trial
            Search: LR ∈ [1e-5, 5e-5] · batch ∈ {8,16,32} · epochs ∈ [3,10]
            Fixed: weight_decay=0.01 · warmup=10% · early_stop patience=2

Final     : Best params + seeds {123, 456, 768} → mean ± std on held-out test set
Metrics   : Overall accuracy + subclass-wise accuracy (HR, AI-R, HF, AI-F)
            + precision, recall, F1 (overall and per-subclass)
Statistics: McNemar's test + Benjamini-Hochberg FDR correction (α=0.05)
Resume    : Colab-safe via Optuna SQLite + JSON progress
Output    : master_results.xlsx — persistent across all runs (Summary,
            Subclass_Matrix, Predictions, Statistical_Tests sheets)
UI        : Terminal CLI with numbered selection menus

Excel columns expected per sheet:
  label     — 0=fake, 1=real  (flipped internally to 0=real, 1=fake)
  article   — article text (already preprocessed)
  topic     — topic metadata (preserved, not used in training)
  news_type — subclass: HR | HF | AI-R | AI-F
  split     — train | val | test
=============================================================================
"""

import time
SCRIPT_START_TIME = time.time()

# =============================================================================
# SECTION 0 · PACKAGE INSTALLATION
# =============================================================================

import subprocess, sys

_PACKAGES = [
    "transformers>=4.38.0",
    "datasets==3.6.0",
    "scikit-learn",
    "scipy",
    "optuna>=3.6.0",
    "accelerate",
    "statsmodels",
    "openpyxl",
]
for _pkg in _PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], capture_output=True)

# =============================================================================
# SECTION 1 · IMPORTS
# =============================================================================

import os, json, random, re, warnings, logging
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# openpyxl styling — imported lazily inside ExcelResultsManager
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =============================================================================
# HF TOKEN SETUP
# =============================================================================
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✓ HF_TOKEN loaded from Colab secrets")
except Exception:
    pass

print("✓ Imports complete")

# =============================================================================
# SECTION 2 · CONFIGURATION
# =============================================================================

SUBCLASSES       = ["HR", "AI-R", "HF", "AI-F"]
SUBCLASS_TO_ID   = {s: i for i, s in enumerate(SUBCLASSES)}
ID_TO_SUBCLASS   = {i: s for i, s in enumerate(SUBCLASSES)}
LABEL_INT_TO_STR = {0: "real", 1: "fake"}

ARCHITECTURES    = ["bert", "distilbert"]


class Config:
    # ── Seeds ──────────────────────────────────────────────────────────────────
    HPO_SEEDS   = [123, 456, 789]
    FINAL_SEEDS = [123, 456, 768]
    DATA_SEED   = 42

    # ── Model HuggingFace IDs ─────────────────────────────────────────────────
    MODELS: Dict[str, str] = {
        "bert":       "jcblaise/bert-tagalog-base-cased",
        "distilbert": "jcblaise/distilbert-tagalog-base-cased",
    }

    # ── Optuna HPO ────────────────────────────────────────────────────────────
    N_TRIALS    = 20
    LR_LOW      = 1e-5
    LR_HIGH     = 5e-5
    BATCH_SIZES = [8, 16, 32]
    EPOCHS_MIN  = 3
    EPOCHS_MAX  = 10

    # ── Fixed training hyperparameters ────────────────────────────────────────
    WEIGHT_DECAY            = 0.01
    WARMUP_RATIO            = 0.10
    GRADIENT_CLIP           = 1.0
    FP16                    = True
    EARLY_STOPPING_PATIENCE = 2

    # ── Sequence ──────────────────────────────────────────────────────────────
    MAX_SEQ_LENGTH = 512

    # ── HuggingFace Hub ───────────────────────────────────────────────────────
    HF_HUB_NAMESPACE   = "nui000"   # ← your HF username or org name
    PUSH_TO_HUB        = True

    # ── Paths ─────────────────────────────────────────────────────────────────
    BASE_DIR           = "/content/drive/MyDrive/filipino_fake_news_hpo"
    MODELS_DIR         = BASE_DIR + "/models"
    RESULTS_DIR        = BASE_DIR + "/results"
    OPTUNA_DB_PATH     = BASE_DIR + "/optuna.db"
    TRIALS_LOG_PATH    = BASE_DIR + "/trials_log.json"
    PROGRESS_PATH      = BASE_DIR + "/progress.json"
    MASTER_EXCEL_PATH  = BASE_DIR + "/results/master_results.xlsx"   # ← persistent

    # ── Statistical testing ───────────────────────────────────────────────────
    FDR_ALPHA = 0.05


config = Config()

# =============================================================================
# SECTION 3 · UTILITIES
# =============================================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


def elapsed(start: float) -> str:
    s = int(time.time() - start)
    h, rem  = divmod(s, 3600)
    m, secs = divmod(rem, 60)
    return f"{h:02d}h {m:02d}m {secs:02d}s"


def model_key(arch: str, sheet_name: str) -> str:
    return f"{arch}__{sheet_name}"


def _parse_mkey(mkey: str) -> Tuple[str, str]:
    arch, sheet_name = mkey.split("__", 1)
    return arch, sheet_name


def all_model_keys(sheet_names: List[str]) -> List[str]:
    return [model_key(a, s) for a in ARCHITECTURES for s in sheet_names]


def mount_drive() -> bool:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("✓ Google Drive mounted")
        return True
    except Exception:
        print("⚠  Running without Google Drive — redirecting paths to ./output")
        _base = "./output"
        config.BASE_DIR           = _base
        config.MODELS_DIR         = _base + "/models"
        config.RESULTS_DIR        = _base + "/results"
        config.OPTUNA_DB_PATH     = _base + "/optuna.db"
        config.TRIALS_LOG_PATH    = _base + "/trials_log.json"
        config.PROGRESS_PATH      = _base + "/progress.json"
        config.MASTER_EXCEL_PATH  = _base + "/results/master_results.xlsx"
        return False


def setup_dirs() -> None:
    for _d in [config.MODELS_DIR, config.RESULTS_DIR]:
        os.makedirs(_d, exist_ok=True)


def check_gpu() -> bool:
    if torch.cuda.is_available():
        _name = torch.cuda.get_device_name(0)
        _mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_name}  ({_mem:.1f} GB)")
        return True
    print("⚠  No GPU — training will be very slow")
    return False


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_TOKENIZER_CACHE: Dict[str, Any] = {}

def get_tokenizer(model_name: str):
    if model_name not in _TOKENIZER_CACHE:
        print(f"  Loading tokenizer: {model_name}")
        _TOKENIZER_CACHE[model_name] = AutoTokenizer.from_pretrained(model_name)
    return _TOKENIZER_CACHE[model_name]


def prefetch_models() -> None:
    for arch, model_name in config.MODELS.items():
        from huggingface_hub import try_to_load_from_cache
        cached = try_to_load_from_cache(model_name, "config.json")
        if cached is not None:
            print(f"  [{arch}] ✓ Already cached — {model_name}")
        else:
            print(f"  [{arch}] Downloading {model_name}…")
        get_tokenizer(model_name)
        _tmp = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=1, problem_type="single_label_classification",
        )
        del _tmp
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"  [{arch}] ✓ Ready")

# =============================================================================
# SECTION 4 · PROGRESS TRACKER & TRIALS LOG
# =============================================================================

class ProgressTracker:
    def __init__(self, path: str) -> None:
        self.path = path
        self.data = self._load()

    def _load(self) -> dict:
        if os.path.exists(self.path):
            with open(self.path) as f:
                return json.load(f)
        return {"hpo_done": {}, "final_done": {}}

    def _save(self) -> None:
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def is_hpo_done(self, key: str) -> bool:
        return self.data["hpo_done"].get(key, False)

    def mark_hpo_done(self, key: str) -> None:
        self.data["hpo_done"][key] = True
        self._save()

    def is_final_done(self, key: str) -> bool:
        return self.data["final_done"].get(key, False)

    def mark_final_done(self, key: str) -> None:
        self.data["final_done"][key] = True
        self._save()

    def n_final_done(self) -> int:
        return sum(1 for v in self.data["final_done"].values() if v)


class TrialsLog:
    def __init__(self, path: str) -> None:
        self.path = path
        self.data = self._load()

    def _load(self) -> dict:
        if os.path.exists(self.path):
            with open(self.path) as f:
                return json.load(f)
        return {}

    def _save(self) -> None:
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2, default=str)

    def add(self, key: str, record: dict) -> None:
        self.data[key] = record
        self._save()

    def get(self, key: str) -> Optional[dict]:
        return self.data.get(key)

    def all_for_model(self, mkey: str) -> List[dict]:
        return sorted(
            [v for k, v in self.data.items() if k.startswith(f"trial_{mkey}_")],
            key=lambda r: r.get("trial_number", 0),
        )

# =============================================================================
# SECTION 5 · DATASET LOADING FROM EXCEL
# =============================================================================

def load_excel(excel_path: str) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Load the pre-processed, pre-stratified Excel dataset.

    Excel columns (case-insensitive): label, article, topic, news_type, split
      label     — 0=fake, 1=real  →  flipped internally to 0=real, 1=fake
      article   — article text (already preprocessed)
      news_type — subclass: HR | HF | AI-R | AI-F
      split     — train | val | test

    Val/test rows are taken from the first sheet (shared held-out set).

    Returns:
      train_dfs   : dict  sheet_name → training DataFrame
      val_df      : shared validation DataFrame
      test_df     : shared test DataFrame
      sheet_names : ordered list of experiment sheet names
    """
    print("\n" + "=" * 70)
    print("DATASET LOADING")
    print("=" * 70)

    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"Excel file not found: {excel_path}")

    print(f"\n  Reading: {excel_path}")
    xl = pd.ExcelFile(excel_path)
    sheet_names = [s for s in xl.sheet_names if s.strip().lower() != "summary"]

    if not sheet_names:
        raise ValueError("No valid experiment sheets found.")

    print(f"  Found {len(sheet_names)} sheet(s): {', '.join(sheet_names)}")

    train_dfs: Dict[str, pd.DataFrame] = {}
    val_df = test_df = None

    for i, sheet in enumerate(sheet_names):
        df = xl.parse(sheet)
        df.columns = df.columns.str.lower().str.strip()
        df = df.rename(columns={"article": "text", "news_type": "subclass"})

        missing = [c for c in ("label", "text", "subclass", "split") if c not in df.columns]
        if missing:
            raise ValueError(f"Sheet '{sheet}' is missing columns: {missing}")

        # Flip label: Excel 0=fake,1=real → internal 0=real,1=fake
        df["label"] = 1 - df["label"].astype(int)
        df["text"]  = df["text"].astype(str)

        if i == 0:
            val_df  = df[df["split"] == "val"].reset_index(drop=True)
            test_df = df[df["split"] == "test"].reset_index(drop=True)

        train_part = df[df["split"] == "train"].reset_index(drop=True)
        train_dfs[sheet] = train_part

        sc_counts = ", ".join(f"{sc}={sum(train_part['subclass'] == sc)}" for sc in SUBCLASSES)
        print(f"  ✓ {sheet}: {len(train_part)} train  ({sc_counts})")

    print(f"  Shared val={len(val_df)}  test={len(test_df)}  (from '{sheet_names[0]}')")
    return train_dfs, val_df, test_df, sheet_names


def build_training_set(train_dfs: Dict[str, pd.DataFrame], sheet_name: str) -> pd.DataFrame:
    return train_dfs[sheet_name].sample(frac=1, random_state=config.DATA_SEED).reset_index(drop=True)

# =============================================================================
# SECTION 6 · PYTORCH DATASET
# =============================================================================

class FakeNewsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 512) -> None:
        texts             = df["text"].astype(str).tolist()
        self.labels       = torch.tensor(df["label"].astype(float).tolist(), dtype=torch.float)
        self.subclass_ids = torch.tensor(
            [SUBCLASS_TO_ID.get(s, 0) for s in df["subclass"].tolist()], dtype=torch.long
        )
        enc = tokenizer(
            texts,
            add_special_tokens=True,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
            "subclass_id":    self.subclass_ids[idx],
        }


_DATASET_CACHE: Dict[str, FakeNewsDataset] = {}

def make_loader(df: pd.DataFrame, tokenizer, batch_size: int, shuffle: bool,
                cache_key: str = "") -> DataLoader:
    if cache_key and cache_key in _DATASET_CACHE:
        ds = _DATASET_CACHE[cache_key]
    else:
        ds = FakeNewsDataset(df, tokenizer, config.MAX_SEQ_LENGTH)
        if cache_key:
            _DATASET_CACHE[cache_key] = ds
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False)

# =============================================================================
# SECTION 7 · TRAINING & EVALUATION
# =============================================================================

CRITERION = nn.BCEWithLogitsLoss()


def train_one_epoch(model, loader, optimizer, scheduler, scaler) -> float:
    model.train()
    total_loss = 0.0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(ids, attention_mask=mask).logits.view(-1)
                loss   = CRITERION(logits, lbls)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(ids, attention_mask=mask).logits.view(-1)
            loss   = CRITERION(logits, lbls)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loader(model, loader) -> Dict[str, Any]:
    """
    Evaluate model on a DataLoader.

    Returns a dict containing:
      loss, accuracy, precision, recall, f1
      subclass_acc, subclass_prec, subclass_rec, subclass_f1  (per-subclass dicts)
      preds        — np.ndarray of int predictions (0/1)
      labels       — np.ndarray of int ground-truth labels (0/1)
      subclass_ids — np.ndarray of subclass indices
      probs        — np.ndarray of sigmoid probabilities [0.0, 1.0]   ← NEW
    """
    model.eval()
    total_loss    = 0.0
    all_preds:    List[int]   = []
    all_labels:   List[int]   = []
    all_subclass: List[int]   = []
    all_probs:    List[float] = []   # ← raw sigmoid scores

    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        sids = batch["subclass_id"]

        logits = model(ids, attention_mask=mask).logits.view(-1)
        total_loss += CRITERION(logits, lbls).item()

        probs_batch = torch.sigmoid(logits).cpu().numpy()
        preds       = (probs_batch >= 0.5).astype(int)

        all_preds.extend(preds.tolist())
        all_labels.extend(lbls.cpu().long().numpy().tolist())
        all_subclass.extend(sids.numpy().tolist())
        all_probs.extend(probs_batch.tolist())   # ← store sigmoid outputs

    preds_arr    = np.array(all_preds)
    labels_arr   = np.array(all_labels)
    subclass_arr = np.array(all_subclass)
    probs_arr    = np.array(all_probs)            # ← shape (N,)

    prec_ov, rec_ov, f1_ov, _ = precision_recall_fscore_support(
        labels_arr, preds_arr, average="binary", zero_division=0
    )

    def _sc_metrics(sc: str):
        sid  = SUBCLASS_TO_ID[sc]
        mask = subclass_arr == sid
        if mask.sum() == 0:
            return None, None, None, None
        p, r, f, _ = precision_recall_fscore_support(
            labels_arr[mask], preds_arr[mask], average="binary", zero_division=0
        )
        acc = float(accuracy_score(labels_arr[mask], preds_arr[mask]))
        return acc, float(p), float(r), float(f)

    subclass_acc, subclass_prec, subclass_rec, subclass_f1 = {}, {}, {}, {}
    for sc in SUBCLASSES:
        a, p, r, f = _sc_metrics(sc)
        subclass_acc[sc]  = round(a, 6) if a is not None else None
        subclass_prec[sc] = round(p, 6) if p is not None else None
        subclass_rec[sc]  = round(r, 6) if r is not None else None
        subclass_f1[sc]   = round(f, 6) if f is not None else None

    return {
        "loss":          round(total_loss / len(loader), 6),
        "accuracy":      round(float(accuracy_score(labels_arr, preds_arr)), 6),
        "precision":     round(float(prec_ov), 6),
        "recall":        round(float(rec_ov),  6),
        "f1":            round(float(f1_ov),   6),
        "subclass_acc":  subclass_acc,
        "subclass_prec": subclass_prec,
        "subclass_rec":  subclass_rec,
        "subclass_f1":   subclass_f1,
        "preds":         preds_arr,
        "labels":        labels_arr,
        "subclass_ids":  subclass_arr,
        "probs":         probs_arr,          # ← NEW: raw sigmoid scores
    }


def full_training_run(
    model_name:      str,
    train_df:        pd.DataFrame,
    val_df:          pd.DataFrame,
    test_df:         Optional[pd.DataFrame],
    hparams:         Dict[str, Any],
    seed:            int,
    train_cache_key: str = "",
    val_cache_key:   str = "",
    test_cache_key:  str = "",
    save_dir:        Optional[str] = None,
    hub_repo_id:     Optional[str] = None,
    trial:           Optional[optuna.Trial] = None,
    seed_index:      int = 0,
    verbose:         bool = True,
) -> Dict[str, Any]:
    set_seed(seed)

    lr         = hparams["learning_rate"]
    batch_size = hparams["batch_size"]
    max_epochs = hparams["max_epochs"]

    tokenizer    = get_tokenizer(model_name)
    train_loader = make_loader(train_df, tokenizer, batch_size, shuffle=True,  cache_key=train_cache_key)
    val_loader   = make_loader(val_df,   tokenizer, batch_size, shuffle=False, cache_key=val_cache_key)
    test_loader  = (
        make_loader(test_df, tokenizer, batch_size, shuffle=False, cache_key=test_cache_key)
        if test_df is not None else None
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1, problem_type="single_label_classification",
    ).to(DEVICE)
    model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY,
        betas=(0.9, 0.999), eps=1e-8,
    )
    n_steps   = len(train_loader) * max_epochs
    n_warmup  = int(n_steps * config.WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=n_warmup, num_training_steps=n_steps
    )
    scaler = (
        torch.cuda.amp.GradScaler()
        if config.FP16 and DEVICE.type == "cuda" else None
    )

    history:         List[dict] = []
    best_val_loss    = float("inf")
    patience_counter = 0
    best_state       = None
    run_start        = time.time()
    test_metrics     = None

    try:
        for epoch in range(1, max_epochs + 1):
            ep_start   = time.time()
            train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
            val_m      = evaluate_loader(model, val_loader)

            ep_rec = {
                "epoch":            epoch,
                "train_loss":       round(train_loss, 6),
                "val_loss":         val_m["loss"],
                "val_accuracy":     val_m["accuracy"],
                "val_f1":           val_m["f1"],
                "val_precision":    val_m["precision"],
                "val_recall":       val_m["recall"],
                "val_subclass_acc": val_m["subclass_acc"],
                "val_subclass_f1":  val_m["subclass_f1"],
                "time_s":           round(time.time() - ep_start, 1),
            }
            history.append(ep_rec)

            if verbose:
                _star  = " ⭐" if val_m["loss"] < best_val_loss else ""
                sc_str = "  ".join(
                    f"{sc}={val_m['subclass_acc'][sc]:.4f}"
                    for sc in SUBCLASSES
                    if val_m["subclass_acc"].get(sc) is not None
                )
                print(
                    f"    Epoch {epoch}/{max_epochs}  "
                    f"loss={val_m['loss']:.4f}  acc={val_m['accuracy']:.4f}  "
                    f"f1={val_m['f1']:.4f}  p={val_m['precision']:.4f}  r={val_m['recall']:.4f}  "
                    + (f"[{sc_str}]  " if sc_str else "")
                    + f"{ep_rec['time_s']:.0f}s{_star}"
                )

            if trial is not None:
                _step = seed_index * config.EPOCHS_MAX + epoch
                trial.report(val_m["accuracy"], step=_step)
                if trial.should_prune():
                    if verbose:
                        print(f"    ✂️  Pruned at epoch {epoch}")
                    raise optuna.TrialPruned()

            if val_m["loss"] < best_val_loss:
                best_val_loss    = val_m["loss"]
                patience_counter = 0
                best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if verbose:
                    print(f"    ⏳ Patience {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
                if patience_counter >= config.EARLY_STOPPING_PATIENCE:
                    if verbose:
                        print(f"    🛑 Early stopping at epoch {epoch}")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        test_metrics = evaluate_loader(model, test_loader) if test_loader is not None else None

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)
            _safe_metrics = {
                k: v for k, v in (test_metrics or {}).items()
                if k not in ("preds", "labels", "subclass_ids", "probs")
            }
            with open(os.path.join(save_dir, "training_meta.json"), "w") as f:
                json.dump({
                    "hparams":      hparams,
                    "seed":         seed,
                    "history":      history,
                    "test_metrics": _safe_metrics,
                }, f, indent=2, default=str)

        if hub_repo_id and config.PUSH_TO_HUB:
            if verbose:
                print(f"    ⬆  Pushing to Hub → {hub_repo_id}")
            try:
                _safe_metrics = {
                    k: v for k, v in (test_metrics or {}).items()
                    if k not in ("preds", "labels", "subclass_ids", "probs")
                }
                model.push_to_hub(hub_repo_id, commit_message=f"seed={seed}")
                tokenizer.push_to_hub(hub_repo_id)
                from huggingface_hub import upload_file
                import tempfile
                with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as tmp:
                    json.dump({
                        "hparams":      hparams,
                        "seed":         seed,
                        "history":      history,
                        "test_metrics": _safe_metrics,
                    }, tmp, indent=2, default=str)
                    tmp_path = tmp.name
                upload_file(
                    path_or_fileobj=tmp_path,
                    path_in_repo="training_meta.json",
                    repo_id=hub_repo_id,
                )
                if verbose:
                    print(f"    ✓ Pushed → https://huggingface.co/{hub_repo_id}")
            except Exception as e:
                print(f"    ⚠  Hub push failed (non-fatal): {e}")

    finally:
        import gc
        del model
        if scaler is not None:
            del scaler
        del optimizer, scheduler
        if best_state is not None:
            del best_state
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

    return {
        "history":       history,
        "test_metrics":  test_metrics,
        "run_time_s":    round(time.time() - run_start, 1),
        "best_val_loss": best_val_loss,
    }

# =============================================================================
# SECTION 8 · OPTUNA HPO
# =============================================================================

def run_hpo_for_model(
    mkey:       str,
    arch:       str,
    sheet_name: str,
    train_df:   pd.DataFrame,
    val_df:     pd.DataFrame,
    progress:   ProgressTracker,
    trials_log: TrialsLog,
) -> Dict[str, Any]:
    if progress.is_hpo_done(mkey):
        study = optuna.load_study(
            study_name=f"fnf_{mkey}",
            storage=f"sqlite:///{config.OPTUNA_DB_PATH}",
        )
        return study.best_trial.params

    model_name = config.MODELS[arch]
    study_name = f"fnf_{mkey}"
    storage    = f"sqlite:///{config.OPTUNA_DB_PATH}"

    study = optuna.create_study(
        study_name=study_name, storage=storage, load_if_exists=True,
        direction="maximize",
        sampler=TPESampler(seed=42),
        pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=1, interval_steps=1),
    )

    n_done      = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    n_remaining = max(0, config.N_TRIALS - n_done)

    if n_done > 0:
        print(f"  ↩  Resuming — {n_done} trial(s) done, {n_remaining} remaining")

    def objective(trial: optuna.Trial) -> float:
        hparams = {
            "learning_rate": trial.suggest_float("learning_rate", config.LR_LOW, config.LR_HIGH, log=True),
            "batch_size":    trial.suggest_categorical("batch_size", config.BATCH_SIZES),
            "max_epochs":    trial.suggest_int("max_epochs", config.EPOCHS_MIN, config.EPOCHS_MAX),
        }

        tnum = trial.number + 1
        print(f"\n  {'─'*66}")
        print(f"  Trial {tnum}/{config.N_TRIALS}  [{mkey}]  "
              f"lr={hparams['learning_rate']:.2e}  "
              f"batch={hparams['batch_size']}  epochs={hparams['max_epochs']}")
        print(f"  {'─'*66}")

        seed_accs = []
        for si, seed in enumerate(config.HPO_SEEDS):
            import gc
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            gc.collect()
            print(f"  ── Seed {seed} " + "─"*52)
            hub_repo_id = (
                f"{config.HF_HUB_NAMESPACE}/{arch}-{sheet_name}-seed{seed}".replace("_", "-")
                if config.PUSH_TO_HUB else None
            )
            res = full_training_run(
                model_name=model_name,
                train_df=train_df, val_df=val_df, test_df=None,
                hparams=hparams, seed=seed,
                train_cache_key=f"{mkey}_train",
                val_cache_key="global_val",
                trial=trial, seed_index=si, verbose=True,
            )
            best_ep  = min(res["history"], key=lambda e: e["val_loss"])
            best_acc = best_ep["val_accuracy"]
            seed_accs.append(best_acc)
            print(f"  Seed {seed} best → acc={best_acc:.4f}  ep={best_ep['epoch']}  ({res['run_time_s']/60:.1f}m)")

        mean_acc = float(np.mean(seed_accs))
        std_acc  = float(np.std(seed_accs))

        trials_log.add(f"trial_{mkey}_{trial.number}", {
            "trial_number": trial.number,
            "mkey":         mkey,
            "hparams":      hparams,
            "seed_accs":    seed_accs,
            "mean_acc":     round(mean_acc, 6),
            "std_acc":      round(std_acc,  6),
        })

        print(f"  Trial {tnum} done  mean_acc={mean_acc:.4f} ± {std_acc:.4f}")
        return mean_acc

    if n_remaining > 0:
        study.optimize(
            objective, n_trials=n_remaining,
            show_progress_bar=False,
            catch=(RuntimeError, torch.cuda.OutOfMemoryError),
        )

    best_trial  = study.best_trial
    best_params = best_trial.params
    print(f"\n  🏆 Best trial #{best_trial.number}  "
          f"acc={best_trial.value:.4f}  "
          f"lr={best_params['learning_rate']:.2e}  "
          f"bs={best_params['batch_size']}  ep={best_params['max_epochs']}")

    progress.mark_hpo_done(mkey)
    return best_params

# =============================================================================
# SECTION 9 · FINAL MODEL TRAINING
# =============================================================================

def train_final_model(
    mkey:         str,
    arch:         str,
    sheet_name:   str,
    train_df:     pd.DataFrame,
    val_df:       pd.DataFrame,
    test_df:      pd.DataFrame,
    best_hparams: Dict[str, Any],
    progress:     ProgressTracker,
    trials_log:   TrialsLog,
) -> Dict[str, Any]:
    if progress.is_final_done(mkey):
        rec = trials_log.get(f"final_{mkey}")
        if rec:
            has_probs = all(
                "probs" in sr and sr["probs"]
                for sr in rec.get("seed_results", [])
            )
            if has_probs:
                return rec
            print(f"  ↩  Cached record for {mkey} is missing 'probs' — retraining…")
            progress.data["final_done"][mkey] = False
            progress._save()

    model_name = config.MODELS[arch]
    save_dir   = os.path.join(config.MODELS_DIR, mkey, "final")

    seed_results = []
    for si, seed in enumerate(config.FINAL_SEEDS):
        import gc
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        gc.collect()
        print(f"\n  {'─'*66}")
        print(f"  Final seed {seed} ({si+1}/{len(config.FINAL_SEEDS)})  "
              f"lr={best_hparams['learning_rate']:.2e}  "
              f"bs={best_hparams['batch_size']}  ep={best_hparams['max_epochs']}")
        print(f"  {'─'*66}")

        res = full_training_run(
            model_name=model_name,
            train_df=train_df, val_df=val_df, test_df=test_df,
            hparams=best_hparams, seed=seed,
            train_cache_key=f"{mkey}_train",
            val_cache_key="global_val",
            test_cache_key="global_test",
            save_dir=(save_dir if si == 0 else None),
            verbose=True,
        )

        tm     = res["test_metrics"]
        sc_str = "  ".join(
            f"{sc}={tm['subclass_acc'][sc]:.4f}"
            for sc in SUBCLASSES
            if tm["subclass_acc"].get(sc) is not None
        )
        print(f"  Seed {seed} test → "
              f"acc={tm['accuracy']:.4f}  f1={tm['f1']:.4f}  "
              f"p={tm['precision']:.4f}  r={tm['recall']:.4f}  "
              + (f"[{sc_str}]" if sc_str else "")
              + f"  ({res['run_time_s']/60:.1f}m)")

        seed_results.append({
            "seed":         seed,
            "history":      res["history"],
            # Scalar metrics (JSON-safe)
            "test_metrics": {k: v for k, v in tm.items()
                             if k not in ("preds", "labels", "subclass_ids", "probs")},
            # Arrays stored separately for Excel export
            "preds":        tm["preds"].tolist(),
            "labels":       tm["labels"].tolist(),
            "subclass_ids": tm["subclass_ids"].tolist(),
            "probs":        tm["probs"].tolist(),        # ← NEW: sigmoid scores
            "run_time_s":   res["run_time_s"],
        })

    def _agg(metric: str):
        vals = [r["test_metrics"][metric] for r in seed_results]
        return round(float(np.mean(vals)), 6), round(float(np.std(vals)), 6)

    def _agg_sc(sc_metric: str):
        result = {}
        for sc in SUBCLASSES:
            vals = [r["test_metrics"][sc_metric][sc]
                    for r in seed_results
                    if r["test_metrics"][sc_metric].get(sc) is not None]
            result[sc] = (
                round(float(np.mean(vals)), 6) if vals else None,
                round(float(np.std(vals)),  6) if vals else None,
            )
        return result

    mean_acc,  std_acc  = _agg("accuracy")
    mean_f1,   std_f1   = _agg("f1")
    mean_prec, std_prec = _agg("precision")
    mean_rec,  std_rec  = _agg("recall")

    sc_acc_agg  = _agg_sc("subclass_acc")
    sc_f1_agg   = _agg_sc("subclass_f1")
    sc_prec_agg = _agg_sc("subclass_prec")
    sc_rec_agg  = _agg_sc("subclass_rec")

    record = {
        "mkey":           mkey,
        "arch":           arch,
        "sheet_name":     sheet_name,
        "model_name":     model_name,
        "hparams":        best_hparams,
        "seed_results":   seed_results,
        "mean_accuracy":  mean_acc,  "std_accuracy":  std_acc,
        "mean_f1":        mean_f1,   "std_f1":        std_f1,
        "mean_precision": mean_prec, "std_precision": std_prec,
        "mean_recall":    mean_rec,  "std_recall":    std_rec,
        "mean_subclass_acc":  {sc: v[0] for sc, v in sc_acc_agg.items()},
        "std_subclass_acc":   {sc: v[1] for sc, v in sc_acc_agg.items()},
        "mean_subclass_f1":   {sc: v[0] for sc, v in sc_f1_agg.items()},
        "std_subclass_f1":    {sc: v[1] for sc, v in sc_f1_agg.items()},
        "mean_subclass_prec": {sc: v[0] for sc, v in sc_prec_agg.items()},
        "std_subclass_prec":  {sc: v[1] for sc, v in sc_prec_agg.items()},
        "mean_subclass_rec":  {sc: v[0] for sc, v in sc_rec_agg.items()},
        "std_subclass_rec":   {sc: v[1] for sc, v in sc_rec_agg.items()},
    }

    trials_log.add(f"final_{mkey}", record)
    progress.mark_final_done(mkey)
    return record

# =============================================================================
# SECTION 10 · STATISTICAL TESTING (McNemar's + Benjamini-Hochberg)
# =============================================================================

def _mcnemar_pval(pi: np.ndarray, pj: np.ndarray, lb: np.ndarray) -> Tuple[float, int, int]:
    b = int(((pi == lb) & (pj != lb)).sum())
    c = int(((pi != lb) & (pj == lb)).sum())
    if b + c == 0:
        return 1.0, b, c
    table  = [[0, b], [c, 0]]
    result = mcnemar(table, exact=(b + c < 25))
    return float(result.pvalue), b, c


def run_mcnemar_bh(final_records: Dict[str, dict]) -> pd.DataFrame:
    keys = list(final_records.keys())
    n    = len(keys)
    all_rows = []

    for i in range(n):
        for j in range(i + 1, n):
            ki, kj = keys[i], keys[j]
            ri = final_records[ki]["seed_results"][0]
            rj = final_records[kj]["seed_results"][0]

            pi     = np.array(ri["preds"])
            pj     = np.array(rj["preds"])
            lb     = np.array(ri["labels"])
            sc_ids = np.array(ri["subclass_ids"])

            p, b, c = _mcnemar_pval(pi, pj, lb)
            all_rows.append(("overall", ki, kj, b, c, p))

            for sc in SUBCLASSES:
                sid  = SUBCLASS_TO_ID[sc]
                mask = sc_ids == sid
                if mask.sum() < 2:
                    continue
                p_sc, b_sc, c_sc = _mcnemar_pval(pi[mask], pj[mask], lb[mask])
                all_rows.append((sc, ki, kj, b_sc, c_sc, p_sc))

    if not all_rows:
        return pd.DataFrame()

    raw_ps = [r[5] for r in all_rows]
    reject, p_adj, _, _ = multipletests(raw_ps, alpha=config.FDR_ALPHA, method="fdr_bh")

    rows = []
    for (scope, ki, kj, b, c, raw_p), adj_p, sig in zip(all_rows, p_adj, reject):
        rows.append({
            "scope": scope, "model_1": ki, "model_2": kj,
            "b": b, "c": c,
            "p_raw":       round(raw_p, 6),
            "p_adj_bh":    round(adj_p, 6),
            "significant": bool(sig),
        })

    result_df = pd.DataFrame(rows)
    out_path  = os.path.join(config.RESULTS_DIR, "mcnemar_bh_results.csv")
    result_df.to_csv(out_path, index=False)
    print(f"  Statistical test results saved → {out_path}")

    overall_df = result_df[result_df["scope"] == "overall"]
    print(f"  Overall: {overall_df['significant'].sum()}/{len(overall_df)} pairs significant  (α={config.FDR_ALPHA} BH)")
    for sc in SUBCLASSES:
        sc_df = result_df[result_df["scope"] == sc]
        if not sc_df.empty:
            print(f"  {sc:<6}: {sc_df['significant'].sum()}/{len(sc_df)} pairs significant")

    sig_df = result_df[result_df["significant"]].sort_values("p_adj_bh")
    if not sig_df.empty:
        print(f"\n  Top significant pairs:")
        print(f"  {'Scope':<8} {'Model 1':<35} {'Model 2':<35} {'p_raw':>9} {'p_adj':>9}")
        print("  " + "─" * 100)
        for _, row in sig_df.head(20).iterrows():
            print(f"  {row['scope']:<8} {row['model_1']:<35} {row['model_2']:<35} "
                  f"{row['p_raw']:>9.6f} {row['p_adj_bh']:>9.6f}")

    return result_df

# =============================================================================
# SECTION 10.5 · EXCEL RESULTS MANAGER
# =============================================================================

class ExcelResultsManager:
    """
    Manages a single persistent master_results.xlsx that accumulates results
    across all past, present, and future pipeline runs without overwriting.

    Sheets
    ------
    Summary          One row per (model_key × seed) + one aggregated row per model.
                     Columns: model_key, arch, sheet_name, seed, row_type,
                     accuracy, precision, recall, f1,
                     HR_acc … AI-F_acc, HR_f1 … AI-F_f1,
                     HR_prec … AI-F_prec, HR_rec … AI-F_rec,
                     lr, batch_size, max_epochs, run_time_s, timestamp

    Subclass_Matrix  One aggregated row per model (mean ± std strings).
                     Useful for the master comparison table in the thesis.

    Predictions      One row per test sample × seed × model.
                     Columns: model_key, arch, sheet_name, seed, sample_index,
                     text_preview (first 160 chars), subclass,
                     true_label (real/fake), pred_label (real/fake),
                     correct (TRUE/FALSE), probability (sigmoid score),
                     timestamp

    Statistical_Tests McNemar + Benjamini-Hochberg results.
                     Columns: scope, model_1, model_2, b, c,
                     p_raw, p_adj_bh, significant, timestamp
    """

    # ── Openpyxl hex colors ──────────────────────────────────────────────────
    _HDR_BG    = "1F4E79"   # dark blue header background
    _HDR_FG    = "FFFFFF"   # white header text
    _ALT_ROW   = "D6E4F0"   # alternating data row tint
    _AGG_BG    = "FFF2CC"   # yellow for aggregated summary rows
    _CORRECT   = "C6EFCE"   # green fill for correct predictions
    _WRONG     = "FFC7CE"   # red fill for wrong predictions
    _SIG       = "FFEB9C"   # amber fill for significant stat tests
    _BORDER_C  = "BDD7EE"   # thin border color

    def __init__(self, path: str) -> None:
        self.path = path

    # ── Internal helpers ─────────────────────────────────────────────────────

    def _load_sheet(self, sheet_name: str) -> pd.DataFrame:
        """Load an existing sheet; return empty DataFrame if absent."""
        if not os.path.exists(self.path):
            return pd.DataFrame()
        try:
            return pd.read_excel(self.path, sheet_name=sheet_name, dtype=str)
        except Exception:
            return pd.DataFrame()

    @staticmethod
    def _coerce_val(v):
        """Convert numpy scalars to native Python types for openpyxl."""
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (np.floating,)):
            return None if np.isnan(v) else float(v)
        if isinstance(v, (np.bool_,)):
            return bool(v)
        if pd.isna(v) if not isinstance(v, (list, dict, np.ndarray)) else False:
            return None
        return v

    def _apply_header_style(self, ws, n_cols: int) -> None:
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        fill   = PatternFill("solid", start_color=self._HDR_BG)
        font   = Font(name="Arial", bold=True, color=self._HDR_FG, size=10)
        align  = Alignment(horizontal="center", vertical="center", wrap_text=True)
        border = Border(
            bottom=Side(border_style="medium", color=self._BORDER_C)
        )
        for col in range(1, n_cols + 1):
            cell = ws.cell(row=1, column=col)
            cell.fill, cell.font, cell.alignment, cell.border = fill, font, align, border

    def _apply_data_style(self, ws, df: pd.DataFrame,
                          color_col: Optional[str] = None,
                          color_true: Optional[str] = None,
                          color_false: Optional[str] = None,
                          highlight_col: Optional[str] = None,
                          highlight_val: Any = None,
                          highlight_color: Optional[str] = None) -> None:
        from openpyxl.styles import Font, PatternFill, Alignment
        font_normal = Font(name="Arial", size=9)
        font_mono   = Font(name="Courier New", size=9)
        align_left  = Alignment(horizontal="left",   vertical="center")
        align_right = Alignment(horizontal="right",  vertical="center")
        align_center= Alignment(horizontal="center", vertical="center")
        fill_alt    = PatternFill("solid", start_color=self._ALT_ROW)
        fill_agg    = PatternFill("solid", start_color=self._AGG_BG)

        color_col_idx      = list(df.columns).index(color_col)      if color_col      and color_col      in df.columns else None
        highlight_col_idx  = list(df.columns).index(highlight_col)  if highlight_col  and highlight_col  in df.columns else None

        for row_idx, (_, row) in enumerate(df.iterrows(), start=2):
            is_agg = (str(row.get("row_type", "")) == "aggregated")
            for col_idx, (col_name, val) in enumerate(row.items(), start=1):
                cell = ws.cell(row=row_idx, column=col_idx)
                is_numeric = isinstance(val, (int, float)) and not isinstance(val, bool)
                cell.font      = font_mono if col_name in ("text_preview",) else font_normal
                cell.alignment = align_right if is_numeric else (
                    align_center if col_name in ("correct", "significant", "seed", "row_type") else align_left
                )
                if is_agg:
                    cell.fill = fill_agg
                elif row_idx % 2 == 0:
                    cell.fill = fill_alt

            # Color-code a boolean column (e.g. "correct", "significant")
            if color_col_idx is not None and color_true and color_false:
                from openpyxl.styles import PatternFill as PF
                raw_val = row.iloc[color_col_idx]
                if str(raw_val).upper() in ("TRUE", "1", "YES"):
                    ws.cell(row=row_idx, column=color_col_idx + 1).fill = PF("solid", start_color=color_true)
                elif str(raw_val).upper() in ("FALSE", "0", "NO"):
                    ws.cell(row=row_idx, column=color_col_idx + 1).fill = PF("solid", start_color=color_false)

            # Highlight rows where a column matches a value
            if highlight_col_idx is not None and highlight_val is not None and highlight_color:
                from openpyxl.styles import PatternFill as PF
                raw_val = row.iloc[highlight_col_idx]
                if str(raw_val).upper() == str(highlight_val).upper():
                    fill_h = PF("solid", start_color=highlight_color)
                    for col_idx in range(1, len(df.columns) + 1):
                        ws.cell(row=row_idx, column=col_idx).fill = fill_h

    def _set_col_widths(self, ws, df: pd.DataFrame, overrides: Optional[Dict[str, int]] = None) -> None:
        overrides = overrides or {}
        for col_idx, col_name in enumerate(df.columns, start=1):
            if col_name in overrides:
                width = overrides[col_name]
            else:
                max_data = max((len(str(v)) for v in df[col_name] if v is not None), default=0)
                width = min(max(max_data + 2, len(col_name) + 2, 8), 55)
            ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = width

    def _write_sheet(self, wb, sheet_name: str, df: pd.DataFrame,
                     freeze: str = "A2",
                     col_widths: Optional[Dict[str, int]] = None,
                     **style_kwargs) -> None:
        from openpyxl.styles import Font, PatternFill, Alignment
        if sheet_name in wb.sheetnames:
            del wb[sheet_name]
        ws = wb.create_sheet(sheet_name)

        # Write header
        for col_idx, col_name in enumerate(df.columns, start=1):
            ws.cell(row=1, column=col_idx, value=col_name)

        # Write data rows
        for row_idx, (_, row) in enumerate(df.iterrows(), start=2):
            for col_idx, val in enumerate(row, start=1):
                ws.cell(row=row_idx, column=col_idx, value=self._coerce_val(val))

        self._apply_header_style(ws, len(df.columns))
        self._apply_data_style(ws, df, **style_kwargs)
        self._set_col_widths(ws, df, overrides=col_widths)

        ws.freeze_panes = freeze
        ws.row_dimensions[1].height = 30

    # ── Public: build DataFrames from a record ────────────────────────────────

    def build_summary_rows(self, record: dict) -> pd.DataFrame:
        """
        Returns a DataFrame with one row per seed + one aggregated row.
        Upserts into the existing Summary sheet (removes old rows for same mkey).
        """
        ts   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        rows = []

        for sr in record["seed_results"]:
            tm = sr["test_metrics"]
            rows.append({
                "model_key":   record["mkey"],
                "arch":        record["arch"].upper(),
                "sheet_name":  record["sheet_name"],
                "seed":        sr["seed"],
                "row_type":    "seed_run",
                # ── Primary metrics ──────────────────────────────────────────
                "accuracy":    tm["accuracy"],
                "precision":   tm["precision"],
                "recall":      tm["recall"],
                "f1":          tm["f1"],
                # ── Subclass accuracy ────────────────────────────────────────
                "HR_acc":      tm["subclass_acc"].get("HR"),
                "AIR_acc":     tm["subclass_acc"].get("AI-R"),
                "HF_acc":      tm["subclass_acc"].get("HF"),
                "AIF_acc":     tm["subclass_acc"].get("AI-F"),
                # ── Subclass F1 ──────────────────────────────────────────────
                "HR_f1":       tm["subclass_f1"].get("HR"),
                "AIR_f1":      tm["subclass_f1"].get("AI-R"),
                "HF_f1":       tm["subclass_f1"].get("HF"),
                "AIF_f1":      tm["subclass_f1"].get("AI-F"),
                # ── Subclass precision ───────────────────────────────────────
                "HR_prec":     tm["subclass_prec"].get("HR"),
                "AIR_prec":    tm["subclass_prec"].get("AI-R"),
                "HF_prec":     tm["subclass_prec"].get("HF"),
                "AIF_prec":    tm["subclass_prec"].get("AI-F"),
                # ── Subclass recall ──────────────────────────────────────────
                "HR_rec":      tm["subclass_rec"].get("HR"),
                "AIR_rec":     tm["subclass_rec"].get("AI-R"),
                "HF_rec":      tm["subclass_rec"].get("HF"),
                "AIF_rec":     tm["subclass_rec"].get("AI-F"),
                # ── Hyperparameters ──────────────────────────────────────────
                "lr":          record["hparams"]["learning_rate"],
                "batch_size":  record["hparams"]["batch_size"],
                "max_epochs":  record["hparams"]["max_epochs"],
                "run_time_s":  sr["run_time_s"],
                "timestamp":   ts,
            })

        def _fmt_agg(mean_val, std_val):
            if mean_val is None:
                return None
            return f"{mean_val:.4f}±{std_val:.4f}"

        rows.append({
            "model_key":   record["mkey"],
            "arch":        record["arch"].upper(),
            "sheet_name":  record["sheet_name"],
            "seed":        "MEAN±STD",
            "row_type":    "aggregated",
            "accuracy":    _fmt_agg(record["mean_accuracy"],  record["std_accuracy"]),
            "precision":   _fmt_agg(record["mean_precision"], record["std_precision"]),
            "recall":      _fmt_agg(record["mean_recall"],    record["std_recall"]),
            "f1":          _fmt_agg(record["mean_f1"],        record["std_f1"]),
            "HR_acc":      _fmt_agg(record["mean_subclass_acc"].get("HR"),    record["std_subclass_acc"].get("HR")),
            "AIR_acc":     _fmt_agg(record["mean_subclass_acc"].get("AI-R"),  record["std_subclass_acc"].get("AI-R")),
            "HF_acc":      _fmt_agg(record["mean_subclass_acc"].get("HF"),    record["std_subclass_acc"].get("HF")),
            "AIF_acc":     _fmt_agg(record["mean_subclass_acc"].get("AI-F"),  record["std_subclass_acc"].get("AI-F")),
            "HR_f1":       _fmt_agg(record["mean_subclass_f1"].get("HR"),     record["std_subclass_f1"].get("HR")),
            "AIR_f1":      _fmt_agg(record["mean_subclass_f1"].get("AI-R"),   record["std_subclass_f1"].get("AI-R")),
            "HF_f1":       _fmt_agg(record["mean_subclass_f1"].get("HF"),     record["std_subclass_f1"].get("HF")),
            "AIF_f1":      _fmt_agg(record["mean_subclass_f1"].get("AI-F"),   record["std_subclass_f1"].get("AI-F")),
            "HR_prec":     _fmt_agg(record["mean_subclass_prec"].get("HR"),   record["std_subclass_prec"].get("HR")),
            "AIR_prec":    _fmt_agg(record["mean_subclass_prec"].get("AI-R"), record["std_subclass_prec"].get("AI-R")),
            "HF_prec":     _fmt_agg(record["mean_subclass_prec"].get("HF"),   record["std_subclass_prec"].get("HF")),
            "AIF_prec":    _fmt_agg(record["mean_subclass_prec"].get("AI-F"), record["std_subclass_prec"].get("AI-F")),
            "HR_rec":      _fmt_agg(record["mean_subclass_rec"].get("HR"),    record["std_subclass_rec"].get("HR")),
            "AIR_rec":     _fmt_agg(record["mean_subclass_rec"].get("AI-R"),  record["std_subclass_rec"].get("AI-R")),
            "HF_rec":      _fmt_agg(record["mean_subclass_rec"].get("HF"),    record["std_subclass_rec"].get("HF")),
            "AIF_rec":     _fmt_agg(record["mean_subclass_rec"].get("AI-F"),  record["std_subclass_rec"].get("AI-F")),
            "lr":          record["hparams"]["learning_rate"],
            "batch_size":  record["hparams"]["batch_size"],
            "max_epochs":  record["hparams"]["max_epochs"],
            "run_time_s":  None,
            "timestamp":   ts,
        })

        return pd.DataFrame(rows)

    def build_predictions_rows(self, record: dict, test_df: pd.DataFrame) -> pd.DataFrame:
        """
        Returns a DataFrame with one row per (sample × seed).
        Includes text preview, true/predicted labels, correct flag, and sigmoid score.
        """
        ts    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        texts = test_df["text"].astype(str).tolist() if test_df is not None else []
        rows  = []

        for sr in record["seed_results"]:
            preds   = sr["preds"]
            labels  = sr["labels"]
            sc_ids  = sr["subclass_ids"]
            probs   = sr.get("probs", [None] * len(preds))

            for i, (pred, label, sc_id, prob) in enumerate(zip(preds, labels, sc_ids, probs)):
                rows.append({
                    "model_key":      record["mkey"],
                    "arch":           record["arch"].upper(),
                    "sheet_name":     record["sheet_name"],
                    "seed":           sr["seed"],
                    "sample_index":   i,
                    "text_preview":   texts[i][:160] if i < len(texts) else "",
                    "subclass":       ID_TO_SUBCLASS.get(int(sc_id), "?"),
                    "true_label":     LABEL_INT_TO_STR.get(int(label), "?"),
                    "pred_label":     LABEL_INT_TO_STR.get(int(pred), "?"),
                    "correct":        bool(int(pred) == int(label)),
                    "probability":    round(float(prob), 6) if prob is not None else None,
                    "timestamp":      ts,
                })

        return pd.DataFrame(rows)

    def build_subclass_matrix_rows(self, record: dict) -> pd.DataFrame:
        """
        One aggregated row per model with all mean±std subclass metrics side-by-side.
        Designed to mirror Table 4 / subclass accuracy matrix from the methodology.
        """
        def _fmt(mean_val, std_val):
            if mean_val is None:
                return "—"
            return f"{mean_val:.4f}±{std_val:.4f}"

        row = {
            "model_key":  record["mkey"],
            "arch":       record["arch"].upper(),
            "sheet_name": record["sheet_name"],
            # Overall
            "overall_acc":  _fmt(record["mean_accuracy"],  record["std_accuracy"]),
            "overall_f1":   _fmt(record["mean_f1"],        record["std_f1"]),
            "overall_prec": _fmt(record["mean_precision"], record["std_precision"]),
            "overall_rec":  _fmt(record["mean_recall"],    record["std_recall"]),
        }
        for sc_key, sc_label in [("HR","HR"), ("AI-R","AIR"), ("HF","HF"), ("AI-F","AIF")]:
            row[f"{sc_label}_acc"]  = _fmt(record["mean_subclass_acc"].get(sc_key),  record["std_subclass_acc"].get(sc_key))
            row[f"{sc_label}_f1"]   = _fmt(record["mean_subclass_f1"].get(sc_key),   record["std_subclass_f1"].get(sc_key))
            row[f"{sc_label}_prec"] = _fmt(record["mean_subclass_prec"].get(sc_key), record["std_subclass_prec"].get(sc_key))
            row[f"{sc_label}_rec"]  = _fmt(record["mean_subclass_rec"].get(sc_key),  record["std_subclass_rec"].get(sc_key))
        row["lr"]         = record["hparams"]["learning_rate"]
        row["batch_size"] = record["hparams"]["batch_size"]
        row["max_epochs"] = record["hparams"]["max_epochs"]
        row["timestamp"]  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return pd.DataFrame([row])

    # ── Public: upsert helpers ────────────────────────────────────────────────

    def _upsert(self, existing: pd.DataFrame, new_rows: pd.DataFrame,
                key_col: str = "model_key") -> pd.DataFrame:
        """Remove rows with same key_col value, append new_rows."""
        if not existing.empty and key_col in existing.columns:
            # Drop old rows for any model_key present in new_rows
            old_keys = set(new_rows[key_col].unique())
            existing = existing[~existing[key_col].isin(old_keys)]
        return pd.concat([existing, new_rows], ignore_index=True)

    def _upsert_predictions(self, existing: pd.DataFrame,
                            new_rows: pd.DataFrame) -> pd.DataFrame:
        """Upsert predictions by (model_key, seed) to allow partial updates."""
        if not existing.empty and "model_key" in existing.columns and "seed" in existing.columns:
            combos = set(zip(new_rows["model_key"], new_rows["seed"].astype(str)))
            mask   = ~existing.apply(
                lambda r: (r["model_key"], str(r["seed"])) in combos, axis=1
            )
            existing = existing[mask]
        return pd.concat([existing, new_rows], ignore_index=True)

    # ── Public: main entry points ─────────────────────────────────────────────

    def update_model_record(self, record: dict, test_df: pd.DataFrame) -> None:
        """
        Call after train_final_model() completes for one model.
        Updates Summary, Subclass_Matrix, and Predictions sheets.
        Does NOT touch Statistical_Tests (call update_stats() separately).
        """
        from openpyxl import load_workbook, Workbook

        print(f"\n  📊 Updating master Excel → {self.path}")

        # ── Load or create workbook ────────────────────────────────────────
        if os.path.exists(self.path):
            try:
                wb = load_workbook(self.path)
            except Exception:
                wb = Workbook()
        else:
            wb = Workbook()
        if "Sheet" in wb.sheetnames:
            del wb["Sheet"]

        # ── Summary ───────────────────────────────────────────────────────
        existing_summary = self._load_sheet("Summary")
        new_summary_rows = self.build_summary_rows(record)
        combined_summary = self._upsert(existing_summary, new_summary_rows)

        self._write_sheet(
            wb, "Summary", combined_summary,
            freeze="D2",
            color_col=None,
            highlight_col="row_type",
            highlight_val="aggregated",
            highlight_color=self._AGG_BG,
            col_widths={"model_key": 42, "sheet_name": 26, "text_preview": 50,
                        "timestamp": 20, "seed": 10},
        )

        # ── Subclass Matrix ───────────────────────────────────────────────
        existing_matrix = self._load_sheet("Subclass_Matrix")
        new_matrix_rows = self.build_subclass_matrix_rows(record)
        combined_matrix = self._upsert(existing_matrix, new_matrix_rows)

        self._write_sheet(
            wb, "Subclass_Matrix", combined_matrix,
            freeze="D2",
            col_widths={"model_key": 42, "sheet_name": 26, "timestamp": 20},
        )

        # ── Predictions ───────────────────────────────────────────────────
        existing_preds = self._load_sheet("Predictions")
        new_pred_rows  = self.build_predictions_rows(record, test_df)
        combined_preds = self._upsert_predictions(existing_preds, new_pred_rows)

        self._write_sheet(
            wb, "Predictions", combined_preds,
            freeze="F2",
            color_col="correct",
            color_true=self._CORRECT,
            color_false=self._WRONG,
            col_widths={"model_key": 42, "sheet_name": 26,
                        "text_preview": 55, "timestamp": 20,
                        "true_label": 12, "pred_label": 12,
                        "correct": 10, "probability": 12},
        )

        # Preserve Statistical_Tests if it already exists
        wb.save(self.path)
        n_correct = int(combined_preds["correct"].apply(lambda v: str(v).upper() == "TRUE").sum()) \
                    if not combined_preds.empty and "correct" in combined_preds.columns else 0
        n_total   = len(combined_preds)
        print(f"  ✓ Summary: {len(combined_summary)} rows  |  "
              f"Predictions: {n_total} rows ({n_correct} correct)  |  "
              f"Subclass_Matrix: {len(combined_matrix)} rows")

    def update_stats(self, stats_df: pd.DataFrame) -> None:
        """
        Call after run_mcnemar_bh() completes.
        Replaces the Statistical_Tests sheet entirely (always reflects latest run).
        """
        if stats_df.empty:
            return
        from openpyxl import load_workbook, Workbook

        if os.path.exists(self.path):
            try:
                wb = load_workbook(self.path)
            except Exception:
                wb = Workbook()
        else:
            wb = Workbook()
        if "Sheet" in wb.sheetnames:
            del wb["Sheet"]

        ts_df = stats_df.copy()
        ts_df["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        self._write_sheet(
            wb, "Statistical_Tests", ts_df,
            freeze="C2",
            color_col="significant",
            color_true=self._SIG,
            color_false=None,
            col_widths={"model_1": 42, "model_2": 42,
                        "scope": 12, "timestamp": 20},
        )

        wb.save(self.path)
        n_sig = int(stats_df["significant"].sum()) if "significant" in stats_df.columns else 0
        print(f"  ✓ Statistical_Tests sheet updated — {n_sig}/{len(stats_df)} significant pairs")

# =============================================================================
# SECTION 11 · TERMINAL CLI
# =============================================================================

def _banner() -> None:
    print()
    print("=" * 70)
    print("  FILIPINO FAKE NEWS DETECTION — HPO TRAINING PIPELINE")
    print("  Cavite State University · 2026")
    print("=" * 70)


def _section(title: str) -> None:
    print(f"\n{'─' * 70}")
    print(f"  {title}")
    print(f"{'─' * 70}")


def _prompt(msg: str, default: str = "") -> str:
    hint = f" [{default}]" if default else ""
    try:
        val = input(f"  {msg}{hint}: ").strip()
    except (EOFError, KeyboardInterrupt):
        print()
        raise
    return val if val else default


def _multiselect(label: str, options: List[str], display_labels: Dict[str, str]) -> List[str]:
    print(f"\n  {label}")
    for i, key in enumerate(options, 1):
        print(f"    [{i}] {display_labels[key]}")
    print(f"    [A] All")

    while True:
        raw = _prompt("Your choice", "A").upper()
        if raw in ("A", "ALL", ""):
            return list(options)
        tokens = re.split(r"[\s,]+", raw)
        chosen, valid = [], True
        for tok in tokens:
            if tok.isdigit():
                idx = int(tok) - 1
                if 0 <= idx < len(options):
                    chosen.append(options[idx])
                else:
                    print(f"  ✗  '{tok}' out of range (1–{len(options)}). Try again.")
                    valid = False; break
            elif tok:
                print(f"  ✗  '{tok}' not valid. Enter numbers or A.")
                valid = False; break
        if valid and chosen:
            return list(dict.fromkeys(chosen))
        if valid and not chosen:
            print("  ✗  No items selected. Try again.")


ARCH_LABELS = {
    "bert":       "Tagalog-BERT        (jcblaise/bert-tagalog-base-cased)",
    "distilbert": "Tagalog-DistilBERT  (jcblaise/distilbert-tagalog-base-cased)",
}


def _fmt(v, width=7) -> str:
    return f"{v:>{width}.4f}" if v is not None else f"{'—':>{width}}"


def _print_grand_summary(final_records: Dict[str, dict]) -> None:
    _section("GRAND SUMMARY — ALL TRAINED MODELS")
    if not final_records:
        print("  No results to display.")
        return

    active_scs = [
        sc for sc in SUBCLASSES
        if any((rec.get("mean_subclass_acc") or {}).get(sc) is not None for rec in final_records.values())
    ]

    col_w = 40
    print()
    hdr = (f"  {'Model':<{col_w}} {'Arch':>12} {'Sheet':>25}  "
           f"{'Acc':>7} {'F1':>7} {'Prec':>7} {'Rec':>7}")
    print(hdr)
    print("  " + "─" * max(len(hdr) - 2, 80))
    for mkey_, rec in sorted(final_records.items(), key=lambda x: x[1].get("mean_f1") or 0, reverse=True):
        print(f"  {mkey_:<{col_w}} {rec.get('arch', '?').upper():>12} {rec.get('sheet_name', '?'):>25}  "
              f"{_fmt(rec.get('mean_accuracy'))}  {_fmt(rec.get('mean_f1'))}  "
              f"{_fmt(rec.get('mean_precision'))}  {_fmt(rec.get('mean_recall'))}")

    if active_scs:
        sc_hdr = "  ".join(f"{sc:>7}" for sc in active_scs)
        print()
        print(f"  Per-subclass accuracy (mean across seeds):")
        hdr2 = f"  {'Model':<{col_w}} {'Arch':>12} {'Sheet':>25}  {sc_hdr}"
        print(hdr2)
        print("  " + "─" * max(len(hdr2) - 2, 80))
        for mkey_, rec in sorted(final_records.items(), key=lambda x: x[1].get("mean_accuracy") or 0, reverse=True):
            sc_vals = "  ".join(_fmt((rec.get("mean_subclass_acc") or {}).get(sc)) for sc in active_scs)
            print(f"  {mkey_:<{col_w}} {rec.get('arch', '?').upper():>12} {rec.get('sheet_name', '?'):>25}  {sc_vals}")

def _step_reset(progress: ProgressTracker, trials_log: TrialsLog, sheet_names: List[str]) -> None:
    _section("RESET TRAINED MODELS")

    all_keys  = all_model_keys(sheet_names)
    done_keys = [k for k in all_keys if progress.is_final_done(k)]
    hpo_only  = [k for k in all_keys if progress.is_hpo_done(k) and not progress.is_final_done(k)]

    if not done_keys and not hpo_only:
        print("  Nothing to reset.")
        return

    if done_keys:
        print(f"  Fully trained ({len(done_keys)}):")
        for k in done_keys: print(f"    • {k}")
    if hpo_only:
        print(f"  HPO only ({len(hpo_only)}):")
        for k in hpo_only: print(f"    • {k}")

    print("\n  [1] Select specific models\n  [2] Reset ALL\n  [3] Cancel")
    choice = _prompt("Your choice", "3")

    if choice == "3":
        print("  Cancelled."); return
    keys_to_reset = (done_keys + hpo_only) if choice == "2" else _multiselect(
        "Select models to reset", done_keys + hpo_only, {k: k for k in done_keys + hpo_only}
    )

    if not keys_to_reset:
        print("  Nothing selected."); return

    ans = _prompt(f"Confirm reset of {len(keys_to_reset)} model(s)? [y/N]", "N").upper()
    if ans not in ("Y", "YES"):
        print("  Cancelled."); return

    for k in keys_to_reset:
        progress.data["hpo_done"].pop(k, None)
        progress.data["final_done"].pop(k, None)
    progress._save()

    for log_key in list(trials_log.data.keys()):
        for mk in keys_to_reset:
            if log_key.startswith(f"trial_{mk}_") or log_key == f"final_{mk}":
                del trials_log.data[log_key]; break
    trials_log._save()

    for k in keys_to_reset:
        try:
            optuna.delete_study(study_name=f"fnf_{k}", storage=f"sqlite:///{config.OPTUNA_DB_PATH}")
        except Exception:
            pass

    print(f"\n  ✓ Reset complete — {len(keys_to_reset)} model(s) cleared.")


def rebuild_master_excel(
    trials_log:   Optional[TrialsLog] = None,
    test_df:      Optional[pd.DataFrame] = None,
    target_keys:  Optional[List[str]] = None,
    run_stats:    bool = True,
) -> None:
    """
    Backfill master_results.xlsx from an existing trials_log.json.

    Safe to call at any time — reads whatever final_* records are present in
    the log, skips any that are missing, and upserts into the Excel file
    without touching records for other models already in the sheet.

    Parameters
    ----------
    trials_log  : TrialsLog instance (auto-loaded from config path if None)
    test_df     : shared test DataFrame for text_preview in Predictions sheet.
                  If None, text_preview cells will be empty — all other columns
                  (labels, preds, probs, subclass) are read from the log.
    target_keys : list of model_key strings to export.
                  If None, all final_* entries in the log are exported.
    run_stats   : whether to re-run McNemar + BH and update Statistical_Tests.

    Typical Colab usage after upgrading the script on an already-trained run
    ─────────────────────────────────────────────────────────────────────────
        rebuild_master_excel(test_df=test_df)

    Or with explicit setup:
        from your_script import TrialsLog, config, rebuild_master_excel
        tl = TrialsLog(config.TRIALS_LOG_PATH)
        rebuild_master_excel(trials_log=tl, test_df=test_df)
    """
    print("\n" + "=" * 70)
    print("  BACKFILL — rebuilding master_results.xlsx from trials_log")
    print("=" * 70)

    if trials_log is None:
        trials_log = TrialsLog(config.TRIALS_LOG_PATH)

    # Collect all final records from log
    all_final_keys = [k for k in trials_log.data if k.startswith("final_")]
    if not all_final_keys:
        print("  ✗  No final_* records found in trials_log. Nothing to export.")
        return

    # Filter to requested keys if specified
    if target_keys is not None:
        wanted_log_keys = {f"final_{mk}" for mk in target_keys}
        all_final_keys  = [k for k in all_final_keys if k in wanted_log_keys]

    if not all_final_keys:
        print("  ✗  None of the requested model keys found in trials_log.")
        return

    print(f"  Found {len(all_final_keys)} record(s) to export:")
    for k in all_final_keys:
        print(f"    • {k.removeprefix('final_')}")

    excel_mgr    = ExcelResultsManager(config.MASTER_EXCEL_PATH)
    final_records: Dict[str, dict] = {}

    for log_key in all_final_keys:
        record = trials_log.data[log_key]
        mkey_  = record.get("mkey", log_key.removeprefix("final_"))

        # Reconstruct arrays from stored lists (if present)
        for sr in record.get("seed_results", []):
            for arr_key in ("preds", "labels", "subclass_ids", "probs"):
                if arr_key in sr and isinstance(sr[arr_key], list):
                    sr[arr_key] = sr[arr_key]   # keep as list; Excel builder accepts lists

        # Auto-patch missing probs from saved checkpoint
        has_probs = all(
            "probs" in sr and sr["probs"]
            for sr in record.get("seed_results", [])
        )
        if not has_probs:
            arch_      = record.get("arch", "")
            sheet_     = record.get("sheet_name", "")
            save_dir   = os.path.join(config.MODELS_DIR, mkey_, "final")
            first_seed = record["seed_results"][0]["seed"] if record.get("seed_results") else 123
            hub_repo   = f"{config.HF_HUB_NAMESPACE}/{arch_}-{sheet_}-seed{first_seed}".replace("_", "-")

            # Prefer Hub, fall back to local checkpoint
            _source = hub_repo if config.PUSH_TO_HUB else (save_dir if os.path.exists(save_dir) else None)

            if arch_ and _source and test_df is not None:
                print(f"  ↩  Patching missing probs for {mkey_} from {_source}…")
                try:
                    _tok   = get_tokenizer(config.MODELS[arch_])
                    _model = AutoModelForSequenceClassification.from_pretrained(_source).to(DEVICE)
                    _bs    = record["hparams"]["batch_size"]
                    _ldr   = make_loader(test_df, _tok, _bs, shuffle=False, cache_key="global_test")
                    _met   = evaluate_loader(_model, _ldr)
                    del _model
                    if DEVICE.type == "cuda":
                        torch.cuda.empty_cache()
                    for sr in record["seed_results"]:
                        if "probs" not in sr or not sr["probs"]:
                            sr["probs"] = _met["probs"].tolist()
                    trials_log.add(log_key, record)
                    print(f"  ✓ Probs patched and log updated for {mkey_}")
                except Exception as e:
                    print(f"  ⚠  Could not patch probs for {mkey_}: {e}")
            else:
                print(f"  ⚠  Skipping probs patch for {mkey_} — checkpoint not found or test_df missing")

        print(f"\n  Exporting {mkey_}…")
        try:
            excel_mgr.update_model_record(record, test_df)
            final_records[mkey_] = record
        except Exception as e:
            print(f"  ⚠  Failed for {mkey_}: {e}")

    # Statistical tests (optional)
    if run_stats and len(final_records) >= 2:
        print("\n  Running McNemar + Benjamini-Hochberg…")
        # Need numpy arrays for McNemar — convert lists back
        records_for_stats = {}
        for mkey_, record in final_records.items():
            r = dict(record)
            r["seed_results"] = []
            for sr in record.get("seed_results", []):
                sr2 = dict(sr)
                for arr_key in ("preds", "labels", "subclass_ids", "probs"):
                    if isinstance(sr2.get(arr_key), list):
                        sr2[arr_key] = np.array(sr2[arr_key])
                r["seed_results"].append(sr2)
            records_for_stats[mkey_] = r
        try:
            stats_df = run_mcnemar_bh(records_for_stats)
            excel_mgr.update_stats(stats_df)
        except Exception as e:
            print(f"  ⚠  Statistical tests failed: {e}")

    print(f"\n  ✓ Done — master Excel at: {config.MASTER_EXCEL_PATH}")

def patch_probs_in_log(mkey, arch, sheet_name, test_df, trials_log):
    """Re-evaluate saved model to backfill 'probs' into an existing log record."""
    model_name = config.MODELS[arch]
    save_dir   = os.path.join(config.MODELS_DIR, mkey, "final")
    rec        = trials_log.get(f"final_{mkey}")
    if not rec or not os.path.exists(save_dir):
        print(f"  ✗  No saved model found at {save_dir}"); return

    tokenizer = get_tokenizer(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(save_dir).to(DEVICE)
    bs        = rec["hparams"]["batch_size"]
    loader    = make_loader(test_df, tokenizer, bs, shuffle=False, cache_key="global_test")
    metrics   = evaluate_loader(model, loader)
    del model
    torch.cuda.empty_cache()

    for sr in rec["seed_results"]:
        if "probs" not in sr or not sr["probs"]:
            sr["probs"] = metrics["probs"].tolist()
            print(f"  ✓ Patched probs for seed {sr['seed']} (from saved seed-0 checkpoint)")

    trials_log.add(f"final_{mkey}", rec)
    print(f"  ✓ Log updated for {mkey}")

def main_cli(selected_keys: Optional[List[str]] = None) -> Dict[str, dict]:
    _banner()

    # ── Setup ─────────────────────────────────────────────────────────────────
    _section("STEP 1 / 4 · PIPELINE SETUP")
    print()
    while True:
        excel_path = _prompt("Excel file path")
        if not excel_path:
            print("  ✗  Path cannot be empty."); continue
        if not os.path.exists(excel_path):
            print(f"  ✗  File not found: {excel_path}"); continue
        break

    mount_drive()
    setup_dirs()
    check_gpu()
    set_seed(42)

    progress    = ProgressTracker(config.PROGRESS_PATH)
    trials_log  = TrialsLog(config.TRIALS_LOG_PATH)
    excel_mgr   = ExcelResultsManager(config.MASTER_EXCEL_PATH)   # ← persistent Excel

    train_dfs, val_df, test_df, sheet_names = load_excel(excel_path)

    print("\n  Pre-downloading model weights…")
    prefetch_models()

    # ── Model selection ───────────────────────────────────────────────────────
    _section("STEP 2 / 4 · MODEL SELECTION")
    if selected_keys is None:
        archs  = _multiselect("ARCHITECTURE", ARCHITECTURES, ARCH_LABELS)
        sheets = _multiselect("EXPERIMENT SHEET", sheet_names, {s: s for s in sheet_names})
        selected_keys = [model_key(a, s) for a in archs for s in sheets]

    keys_to_train = [k for k in selected_keys if not progress.is_final_done(k)]
    n_already     = len(selected_keys) - len(keys_to_train)

    if n_already:
        print(f"\n  ℹ  {n_already} model(s) already trained.")
        ans = _prompt("Reset any? [1=Yes / 2=No]", "2")
        if ans == "1":
            _step_reset(progress, trials_log, sheet_names)
            keys_to_train = [k for k in selected_keys if not progress.is_final_done(k)]

    if not keys_to_train:
        print("\n  ✓ All selected models already trained.\n")
        final_records = {k: trials_log.get(f"final_{k}") for k in selected_keys if trials_log.get(f"final_{k}")}
        _print_grand_summary(final_records)

        # Offer to export to master Excel even though no new training happened
        print("\n  Options:")
        print("    [1] Export these results to master Excel  (master_results.xlsx)")
        print("    [2] Exit")
        ans = _prompt("Your choice", "1")
        if ans == "1":
            rebuild_master_excel(
                trials_log=trials_log,
                test_df=test_df,
                target_keys=list(final_records.keys()),
                run_stats=(len(final_records) >= 2),
            )

        return final_records

    # ── Confirm ───────────────────────────────────────────────────────────────
    _section("STEP 3 / 4 · CONFIRM & TRAIN")
    print(f"  Models to train : {len(keys_to_train)}")
    print(f"  Already done    : {progress.n_final_done()}")
    print(f"  Master Excel    : {config.MASTER_EXCEL_PATH}")
    ans = _prompt("Proceed? [Y/n]", "Y").upper()
    if ans not in ("Y", "YES", ""):
        print("\n  Aborted.\n"); return {}

    # ── Training ──────────────────────────────────────────────────────────────
    _section("STEP 4 / 4 · TRAINING")
    total = len(keys_to_train)
    final_records = {}

    for idx, mkey_ in enumerate(keys_to_train):
        arch_, sheet_ = _parse_mkey(mkey_)

        print(f"\n{'#' * 70}")
        print(f"  [{idx+1}/{total}]  {mkey_}  ({arch_.upper()} | Sheet: {sheet_})")
        print(f"{'#' * 70}")

        train_df_ = build_training_set(train_dfs, sheet_)
        sc_counts = ", ".join(f"{sc}={sum(train_df_['subclass'] == sc)}" for sc in SUBCLASSES)
        print(f"  Training set: {len(train_df_)} samples  ({sc_counts})")

        print(f"\n  HPO — {config.N_TRIALS} trials, seeds {config.HPO_SEEDS}…")
        best_hp = run_hpo_for_model(mkey_, arch_, sheet_, train_df_, val_df, progress, trials_log)

        print(f"\n  Final training — seeds {config.FINAL_SEEDS}…")
        record = train_final_model(
            mkey_, arch_, sheet_, train_df_, val_df, test_df,
            best_hp, progress, trials_log,
        )
        final_records[mkey_] = record

        sc_str = "  ".join(
            f"{sc}={record['mean_subclass_acc'][sc]:.4f}"
            for sc in SUBCLASSES if record["mean_subclass_acc"].get(sc) is not None
        )
        print(f"\n  ✓ {mkey_}  "
              f"acc={record['mean_accuracy']:.4f}±{record['std_accuracy']:.4f}  "
              f"f1={record['mean_f1']:.4f}±{record['std_f1']:.4f}"
              + (f"\n    subclass: {sc_str}" if sc_str else ""))

        # ── Write to master Excel immediately after each model ────────────────
        try:
            excel_mgr.update_model_record(record, test_df)
        except Exception as e:
            print(f"  ⚠  Excel update failed (non-fatal): {e}")

    _print_grand_summary(final_records)

    # ── Statistical tests ─────────────────────────────────────────────────────
    if len(final_records) >= 2:
        print("\n  Running McNemar + Benjamini-Hochberg statistical tests…")
        stats_df = run_mcnemar_bh(final_records)
        try:
            excel_mgr.update_stats(stats_df)
        except Exception as e:
            print(f"  ⚠  Stats sheet update failed (non-fatal): {e}")

    # ── Save JSON results ─────────────────────────────────────────────────────
    out = os.path.join(config.RESULTS_DIR, "final_results.json")
    with open(out, "w") as f:
        json.dump(
            {k: {kk: vv for kk, vv in v.items() if kk not in ("preds", "labels")}
             for k, v in final_records.items()},
            f, indent=2, default=str,
        )
    print(f"\n  ✓ JSON results saved    → {out}")
    print(f"  ✓ Master Excel          → {config.MASTER_EXCEL_PATH}")
    print(f"  Total runtime : {elapsed(SCRIPT_START_TIME)}")
    return final_records


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    main_cli()

✓ HF_TOKEN loaded from Colab secrets
✓ Imports complete

  FILIPINO FAKE NEWS DETECTION — HPO TRAINING PIPELINE
  Cavite State University · 2026

──────────────────────────────────────────────────────────────────────
  STEP 1 / 4 · PIPELINE SETUP
──────────────────────────────────────────────────────────────────────

